# 🏀 HoopAI — 訓練/匯出偵測模型 (Colab)

產出 `hoopai-det.tflite`(球 + 籃框偵測),丟進 HoopAI app 就能真的用相機判進球/放槍。
同一個 `.tflite` 同時給 Android(GPU delegate)和 iOS(CoreML delegate)用。

**先做這件事:** 上方選單 `Runtime → Change runtime type → T4 GPU`,再開始。

有兩條路,建議先跑 A 驗證管線,再跑 B 拿正式模型:

| | Path A(快) | Path B(準) |
|---|---|---|
| 時間 | ~10 分 | ~1–2 小時 |
| 類別 | 2 類(球+籃框) | 4 類(球/籃框/進球/人) |
| 帳號 | 不用 | 免費 Roboflow API key |
| 精度 | 粗略、驗證用 | ~90%(乾淨 CC BY 4.0 授權) |

跑完把下載到的 `hoopai-det.tflite` 給 Claude,我會塞進 app + 重建 APK/IPA。

In [ ]:
# 安裝工具(約 1–2 分)
!pip -q install ultralytics roboflow
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Path A — 快速真模型(2 類:球 + 籃框)
用 avishah3 現成的 6MB YOLOv8-nano 權重,轉成 FP16 TFLite。**不用 GPU、不用帳號。**

⚠️ 這是 **2 類** 模型 → 拿給 Claude 時說「用 Path A 模型」,我會把 `CLASS_ORDER` 暫時改成 `['ball','rim']`。

In [ ]:
import urllib.request, glob, shutil
from ultralytics import YOLO

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/avishah3/AI-Basketball-Shot-Detection-Tracker/master/best.pt',
    'best.pt')
m = YOLO('best.pt')
print('模型類別:', m.names)   # 應該是 {0:'Basketball', 1:'Basketball Hoop'} 這種

# FP16 TFLite,640,NMS 留在 app 的 JS parser(nms=False);不要 int8(Android GPU delegate 不支援)
m.export(format='tflite', imgsz=640, half=True, nms=False)

f = glob.glob('best_saved_model/*float16.tflite') or glob.glob('best_saved_model/*.tflite')
print('產出:', f)
shutil.copy(f[0], 'hoopai-det.tflite')
print('大小(KB):', __import__('os').path.getsize('hoopai-det.tflite')//1024)

from google.colab import files
files.download('hoopai-det.tflite')   # 下載到你電腦

## Path B — 正式 90% 模型(4 類,乾淨授權)
訓練 YOLO11n(nano),資料集用 Roboflow Universe 上 **CC BY 4.0** 的籃球資料集(含 `ball_in_basket`)。

### 步驟
1. 去 [roboflow.com](https://roboflow.com) 註冊(免費)→ 右上頭像 → **Settings → API Keys** 複製你的 key。
2. 開一個籃球偵測資料集,例如:
   - https://universe.roboflow.com/ownense/basketball-detection-3-fjfnq (含 ball/hoop/made 類)
   - https://universe.roboflow.com/sc-xqmxu/basketball-and-net-detection
   - 或搜尋 `basketball detection` 找 class 有「ball / hoop(basket) / made」的
3. 在資料集頁 **Download Dataset → 選 YOLOv11 → Show download code**,把它給你的 `workspace / project / version` 填進下面。

In [ ]:
# ↓↓↓ 填你自己的 key 和資料集(從 Roboflow 的 download code 複製)↓↓↓
ROBOFLOW_API_KEY = 'YOUR_ROBOFLOW_API_KEY'
WORKSPACE        = 'WORKSPACE_SLUG'
PROJECT          = 'PROJECT_SLUG'
VERSION          = 1
# ↑↑↑ ------------------------------------------------ ↑↑↑

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov11')
print('下載到:', dataset.location)

In [ ]:
# 把資料集類別重新對映成 HoopAI 的順序:[ball, rim, ball_in_basket, person]
import yaml, glob, os

yml_path = os.path.join(dataset.location, 'data.yaml')
d = yaml.safe_load(open(yml_path))
names = d['names'] if isinstance(d['names'], list) else [d['names'][i] for i in range(len(d['names']))]
print('資料集原始類別:', list(enumerate(names)))

TARGET = ['ball', 'rim', 'ball_in_basket', 'person']

# ↓↓↓ 依上面印出的原始類別,填「原index → 目標index」;不要的填 None ↓↓↓
# 例:原始 = [Ball, Ball in Basket, Player, Basket, Player-Shooting]
#   REMAP = {0:0, 1:2, 2:3, 3:1, 4:None}
REMAP = {0: 0, 1: 1}   # <-- 改成你資料集的對映!
# ↑↑↑ ---------------------------------------------------------- ↑↑↑

def remap_split(split):
    lbl_dir = os.path.join(dataset.location, split, 'labels')
    if not os.path.isdir(lbl_dir):
        return 0
    changed = 0
    for fn in glob.glob(os.path.join(lbl_dir, '*.txt')):
        out = []
        for line in open(fn):
            p = line.split()
            if not p:
                continue
            src = int(p[0])
            tgt = REMAP.get(src, None)
            if tgt is None:
                continue
            out.append(' '.join([str(tgt)] + p[1:]))
        open(fn, 'w').write('\n'.join(out) + ('\n' if out else ''))
        changed += 1
    return changed

for s in ['train', 'valid', 'test']:
    print(s, '->', remap_split(s), '個標註檔已重映')

d['names'] = TARGET
d['nc'] = len(TARGET)
yaml.safe_dump(d, open(yml_path, 'w'))
print('新 data.yaml 類別:', TARGET)

In [ ]:
# 訓練 YOLO11n(T4 大約 45–90 分,epochs 可先設 100 試)
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data=yml_path, epochs=150, imgsz=640, batch=16, patience=30)
print('最佳權重:', 'runs/detect/train/weights/best.pt')

In [ ]:
# 匯出 FP16 TFLite(+ 可選 CoreML)並下載
import glob, shutil, os
from ultralytics import YOLO
best = 'runs/detect/train/weights/best.pt'
mm = YOLO(best)
print('驗證指標:'); mm.val(data=yml_path)

mm.export(format='tflite', imgsz=640, half=True, nms=False)
f = glob.glob('runs/detect/train/weights/best_saved_model/*float16.tflite') or glob.glob('**/best_saved_model/*.tflite', recursive=True)
shutil.copy(f[0], 'hoopai-det.tflite')
print('TFLite 大小(KB):', os.path.getsize('hoopai-det.tflite')//1024)

# iOS CoreML(可選,fast-tflite 其實也能直接跑上面的 tflite;要更好 ANE 效能才需要)
try:
    mm.export(format='coreml', imgsz=640, half=True, nms=False)
    print('CoreML 已匯出')
except Exception as e:
    print('CoreML 匯出略過(非必要):', e)

from google.colab import files
files.download('hoopai-det.tflite')

---
跑完把 `hoopai-det.tflite` 給 Claude。我會:放進 `assets/models/`、對映類別、觸發 CI 重建 APK/IPA,你再重新側載一次就能真的用相機偵測了。

**要真的到 90%:** 公開資料集訓出來在你自己球場的角度會掉一點。之後可以在 Roboflow 標你自己 50–150 張側拍畫面再 fine-tune 20 epochs,那個才是「你球場的 90%」。